In [2]:
import pandas as pd
import os
import glob

# Find the heart-rate export in the repository raw-data directory.
# This makes the script more robust if the timestamp in the filename changes.
try:
    data_dir = '../../data/raw/samsung_health/export_20260516'
    heart_rate_files = glob.glob(os.path.join(data_dir, 'com.samsung.shealth.tracker.heart_rate.*.csv'))
    if not heart_rate_files:
        raise FileNotFoundError(f"No heart-rate CSV file found in {data_dir}.")
    INPUT_FILE = heart_rate_files[0] # Use the first match
    print(f"Found input file: {INPUT_FILE}")
except FileNotFoundError as e:
    print(e)
    INPUT_FILE = None # Set to None if not found to avoid further errors

OUTPUT_FILE = "../../data/processed/daily_heart_rate_summary.csv"

START_TIME_COL = "com.samsung.health.heart_rate.start_time"
HEART_BEAT_COUNT_COL = "com.samsung.health.heart_rate.heart_beat_count"
PKG_NAME_COL = "com.samsung.health.heart_rate.pkg_name"
HEART_RATE_COL = "com.samsung.health.heart_rate.heart_rate"
MAX_COL = "com.samsung.health.heart_rate.max"
MIN_COL = "com.samsung.health.heart_rate.min"
DATAUUID_COL = "com.samsung.health.heart_rate.datauuid"
CLIENT_DATA_ID_COL = "com.samsung.health.heart_rate.client_data_id"


def build_daily_summary(input_file: str = INPUT_FILE, output_file: str = OUTPUT_FILE) -> pd.DataFrame:
    if not input_file or not os.path.exists(input_file):
        raise FileNotFoundError(f"Input file not found or is invalid: {input_file}")
        
    df = pd.read_csv(input_file, skiprows=1, dtype=str, low_memory=False)

    time_candidate_cols = [START_TIME_COL, HEART_BEAT_COUNT_COL, PKG_NAME_COL]
    time_scores: dict[str, int] = {}
    datetime_candidates: dict[str, pd.Series] = {}

    for col in time_candidate_cols:
        if col in df.columns:
            dt_series = pd.to_datetime(df[col], errors="coerce")
            time_scores[col] = int(dt_series.notna().sum())
            datetime_candidates[col] = dt_series

    if not time_scores:
        raise ValueError("No time-related columns were found in the input file.")

    selected_time_col = max(time_scores, key=time_scores.get)
    if time_scores[selected_time_col] == 0:
        raise ValueError("Could not find parseable datetime values in candidate time columns.")

    hr_candidate_cols = [HEART_RATE_COL, MAX_COL, MIN_COL, DATAUUID_COL, CLIENT_DATA_ID_COL]
    hr_scores: dict[str, int] = {}
    numeric_candidates: dict[str, pd.Series] = {}

    for col in hr_candidate_cols:
        if col in df.columns:
            numeric_series = pd.to_numeric(df[col], errors="coerce")
            hr_scores[col] = int(numeric_series.notna().sum())
            numeric_candidates[col] = numeric_series

    if not hr_scores:
        raise ValueError("No heart-rate related columns were found in the input file.")

    selected_hr_col = max(hr_scores, key=hr_scores.get)
    if hr_scores[selected_hr_col] == 0:
        raise ValueError("Could not find numeric heart-rate values in candidate columns.")

    df = df.copy()
    df[START_TIME_COL] = datetime_candidates[selected_time_col]
    df["heart_rate_value"] = numeric_candidates[selected_hr_col]

    df = df.dropna(subset=[START_TIME_COL, "heart_rate_value"])

    df["date"] = df[START_TIME_COL].dt.date

    daily_summary = (
        df.groupby("date", as_index=False)["heart_rate_value"]
        .agg(
            avg_heart_rate="mean",
            max_heart_rate="max",
            min_heart_rate="min",
        )
        .sort_values("date")
    )

    daily_summary["avg_heart_rate"] = daily_summary["avg_heart_rate"].round(2)

    daily_summary.to_csv(output_file, index=False, encoding="utf-8-sig")
    return daily_summary


if __name__ == "__main__":
    if INPUT_FILE:
        summary = build_daily_summary()
        print(summary.head(10).to_string(index=False))
        print(f"\nSaved: {OUTPUT_FILE}")
        print(f"Rows: {len(summary)}")
    else:
        print("Script did not run because the input file was not found.")


Found input file: New Data\com.samsung.shealth.tracker.heart_rate.20260516153602.csv
      date  avg_heart_rate  max_heart_rate  min_heart_rate
2026-02-06           90.50           177.0            62.0
2026-02-09           72.76           103.0            57.0
2026-02-10           75.81           114.0            45.0
2026-02-11           77.41           163.0            50.0
2026-02-12           77.57           123.0            49.0
2026-02-13           76.37           131.0            48.0
2026-02-14           81.93           137.0            45.0
2026-02-15           76.61           112.0            54.0
2026-02-16           79.66           123.0            49.0
2026-02-17           75.51           124.0            49.0

Saved: daily_heart_rate_summary.csv
Rows: 97
